In [1]:
from transformers import RobertaTokenizer, RobertaForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, LineByLineTextDataset, RobertaConfig
from transformers import Trainer
import torch
from typing import Dict, Union, Any
from datasets import load_dataset

In [32]:
def build_freeze_dict(model, mask_strategy, cutoff_perc, init_model):
    freeze_dict = dict()
    for name, param in model.named_parameters():
        if name[8:15] == "encoder" and name[-6:] == "weight" and name[-16:-7] != "LayerNorm":
            if mask_strategy == "raw_value":
                cutoff = torch.quantile(torch.abs(param), cutoff_perc)
                zero_mask = torch.abs(param) > cutoff
            
            elif mask_strategy == "movement":
                init_param = init_model.state_dict()[name]
                param_change = torch.abs(init_param-param)
                cutoff = torch.quantile(param_change, cutoff_perc)
                zero_mask = torch.abs(param_change) > cutoff
                
            elif mask_strategy == "direction":
                init_param = init_model.state_dict()[name]
                #is the final value closer to zero than the initial value
                zero_mask = torch.abs(init_param)-torch.abs(param) > 0
            freeze_dict[name] = zero_mask
    return freeze_dict

In [33]:
class FreezeTrainer(Trainer):
    def training_step(self, model: torch.nn.Module, inputs: Dict[str, Union[torch.Tensor, Any]]):
        #custom training step that includes zeroing the gradient for masked weights
        model.train()
        inputs = self._prepare_inputs(inputs)
        loss = self.compute_loss(model, inputs)
        loss.backward()
        
        for name, param in model.named_parameters():
            if name in freeze_dict:
                param.grad *= freeze_dict[name] #hard-coded, not sure how to pass a new argument through the Trainer...
        return loss.detach()

In [29]:
#build out tiny model
config = RobertaConfig.from_pretrained("roberta-base")
config.num_attention_heads = 2
config.num_hidden_layers= 2
m = RobertaForMaskedLM(config = config)
m2 = RobertaForMaskedLM(config = config)

/home/anna/miniconda3/envs/fairseq_May24/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [34]:
#build out masks
freeze_dict = build_freeze_dict(m, "raw_value", 0.7, m2)
# freeze_dict_move = build_freeze_dict(m, "movement", 0.7, m2)
# freeze_dict_dir = build_freeze_dict(m, "direction", 0.7, m2)

In [35]:
for name, param in m.named_parameters():
    if name in freeze_dict:
        p = param.detach()
        p *= freeze_dict[name]
        param = p

In [38]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
datacollator = DataCollatorForLanguageModeling(
                tokenizer=tokenizer,
                mlm=True,
                mlm_probability=0.15,
                )

training_args = TrainingArguments(
    output_dir="./freeze_trial",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    save_steps=10_000,
    save_total_limit=2,
    prediction_loss_only=True)

dataset = LineByLineTextDataset(
    tokenizer = tokenizer,
    file_path="/media/anna/Samsung_T5/Initialization/BabyLM/datasets/babyLM_10M/aochildes.train",
    block_size = 128)

In [10]:
trainer = FreezeTrainer(
    model = m,
    args = training_args,
    data_collator=datacollator,
    train_dataset=dataset)

trainer.train()

Step,Training Loss
500,6.609500


KeyboardInterrupt: 

In [40]:
c = None

trainer = FreezeTrainer(
    model = c if c is not None else m,
    args = training_args,
    data_collator=datacollator,
    train_dataset=dataset)

trainer.train()

Step,Training Loss


KeyboardInterrupt: 

In [11]:
for name, param in m.named_parameters():
    if name in freeze_dict:
        print(param)

Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0214,  ...,  0.0000,  0.0211, -0.0000],
        [ 0.0000, -0.0249,  0.0000,  ...,  0.0000,  0.0287, -0.0237],
        [-0.0000,  0.0000,  0.0159,  ...,  0.0000,  0.0000, -0.0000],
        ...,
        [-0.0000, -0.0230, -0.0332,  ...,  0.0000,  0.0000,  0.0205],
        [ 0.0000,  0.0000, -0.0000,  ...,  0.0000,  0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0342]],
       requires_grad=True)
Parameter containing:
tensor([[-0.0000, -0.0324,  0.0000,  ...,  0.0318,  0.0000, -0.0000],
        [-0.0000,  0.0000,  0.0559,  ...,  0.0367,  0.0000, -0.0000],
        [ 0.0000, -0.0373,  0.0000,  ..., -0.0000, -0.0000,  0.0000],
        ...,
        [-0.0000, -0.0000, -0.0480,  ..., -0.0000,  0.0000, -0.0443],
        [-0.0000, -0.0000, -0.0000,  ...,  0.0000,  0.0363, -0.0000],
        [ 0.0342,  0.0000,  0.0000,  ..., -0.0000,  0.0498,  0.0458]],
       requires_grad=True)
Parameter containing:
tensor([[ 0.

# IT WORKS!!!

In [14]:
dataset = load_dataset(path = "./baseline-pretraining/src/babylm_baseline_train/datasets/babyLM_for_hf.py",
                       name = 'babyLM-10M',
                       split = "train")

In [5]:
dataset.column_names

['text']

In [15]:
def preprocess_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, return_tensors="pt")

tokenized_datasets = dataset.map(preprocess_function)

Map:   0%|          | 0/1058740 [00:00<?, ? examples/s]

In [16]:
tokenized_datasets.column_names

['text', 'input_ids', 'attention_mask']

In [26]:
from babylm_baseline_train.datasets import babyLM

baby_dataset = babyLM.get_babyLM_10M(seq_len=128, tokenizer=None, just_dataset=False)

/home/anna/miniconda3/envs/fairseq_May24/lib/python3.9/site-packages/datasets/load.py:929: FutureWarning: The repository for babyLM_for_hf contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/anna/miniconda3/envs/fairseq_May24/lib/python3.9/site-packages/babylm_baseline_train/datasets/babyLM_for_hf.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Map:   0%|          | 0/1058740 [00:00<?, ? examples/s]

Grouping texts with default mode without padding or stride at context length of 128


Map:   0%|          | 0/1058740 [00:00<?, ? examples/s]

In [27]:
baby_dataset.column_names

['input_ids', 'attention_mask', 'labels']